In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression,ElasticNet,LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import VotingClassifier, VotingRegressor, BaggingRegressor, RandomForestClassifier, \
    RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss, r2_score
#from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import make_column_selector
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from tqdm import tqdm
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [39]:
sonar=pd.read_csv('Sonar.csv')
le=LabelEncoder()
y=sonar['Class']
y=le.fit_transform(y)
X=sonar.drop('Class',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [3]:
dtc=DecisionTreeClassifier(random_state=25,max_depth=1)
knn=KNeighborsClassifier()
nb=GaussianNB()
lor=LogisticRegression()

In [4]:
ada=AdaBoostClassifier(estimator=dtc,n_estimators=10,random_state=25)
ada.fit(X_train,y_train)
y_pred=ada.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.73      0.88      0.80        34
           1       0.82      0.62      0.71        29

    accuracy                           0.76        63
   macro avg       0.77      0.75      0.75        63
weighted avg       0.77      0.76      0.76        63



In [5]:
n_est=[10,15,25,50,75,100]
scores=[]
for n in n_est:
    ada=AdaBoostClassifier(estimator=dtc,n_estimators=n,random_state=25)
    ada.fit(X_train,y_train)
    y_pred=ada.predict(X_test)
    y_pred_prob=ada.predict_proba(X_test)
    scores.append([n, accuracy_score(y_test,y_pred),log_loss(y_test,y_pred_prob),r2_score(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['n_estimators','accuracy_score','log_loss','r2_score'])
df_scores.sort_values(ascending=True,by=['log_loss'],inplace=True)
df_scores

,n_estimators,accuracy_score,log_loss,r2_score
0,10,0.761905,0.539927,0.041582
2,25,0.777778,0.561139,0.105477
1,15,0.730159,0.564901,-0.086207
4,75,0.761905,0.579844,0.041582
3,50,0.761905,0.580248,0.041582
5,100,0.761905,0.580572,0.041582


In [6]:
n_est=[10,15,25,50,75,100]
ests=[dtc,nb,lor]
scores=[]
for e in tqdm(ests):
    for n in n_est:
        ada=AdaBoostClassifier(estimator=e,n_estimators=n,random_state=25)
        ada.fit(X_train,y_train)
        y_pred=ada.predict(X_test)
        y_pred_prob=ada.predict_proba(X_test)
        scores.append([e,n,log_loss(y_test,y_pred_prob)])
df_scores=pd.DataFrame(scores,columns=['Estimator','N Estimator','log_loss'])
df_scores.sort_values(ascending=True,by=['log_loss'],inplace=True)
df_scores

100%|██████████| 3/3 [00:02<00:00,  1.22it/s]


,Estimator,N Estimator,log_loss
9,GaussianNB(),50,0.497604
10,GaussianNB(),75,0.501961
11,GaussianNB(),100,0.506158
8,GaussianNB(),25,0.506931
6,GaussianNB(),10,0.520720
15,LogisticRegression(),50,0.537005
17,LogisticRegression(),100,0.537005
16,LogisticRegression(),75,0.537005
13,LogisticRegression(),15,0.537122
0,"DecisionTreeClassifier(max_depth=1, random_sta...",10,0.539927


In [7]:
concrete=pd.read_csv('Concrete_Data.csv')
y=concrete['Strength']
y=le.fit_transform(y)
X=concrete.drop('Strength',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)
# ohe=OneHotEncoder()
# X_train=ohe.fit_transform(X_train)
# X_test=ohe.transform(X_test)
ss=StandardScaler()
X_train=ss.fit_transform(X_train)
X_test=ss.transform(X_test)

In [13]:
dtr=DecisionTreeRegressor(random_state=25,max_depth=1)
lr=LinearRegression()
el=ElasticNet(random_state=25)
n_est = [10,15,25,30,50]
ests = [dtr,el,lr]
scores = []
for e in tqdm(ests):
    for n in n_est:
        ada = AdaBoostRegressor(estimator=e, n_estimators=n, random_state=25)
        ada.fit(X_train, y_train)
        y_pred = ada.predict(X_test)
        scores.append([e, n, r2_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['Estimator', 'N Estimator', 'R2 Score'])
df_scores.sort_values(ascending=True, by=['R2 Score'], inplace=True)
df_scores

100%|██████████| 3/3 [00:00<00:00,  4.88it/s]


,Estimator,N Estimator,R2 Score
0,"DecisionTreeRegressor(max_depth=1, random_stat...",10,0.489654
1,"DecisionTreeRegressor(max_depth=1, random_stat...",15,0.503189
2,"DecisionTreeRegressor(max_depth=1, random_stat...",25,0.520781
5,ElasticNet(random_state=25),10,0.522008
7,ElasticNet(random_state=25),25,0.522008
6,ElasticNet(random_state=25),15,0.522008
9,ElasticNet(random_state=25),50,0.522008
8,ElasticNet(random_state=25),30,0.522008
3,"DecisionTreeRegressor(max_depth=1, random_stat...",30,0.522447
4,"DecisionTreeRegressor(max_depth=1, random_stat...",50,0.539743


In [23]:
sonar=pd.read_csv('Sonar.csv')
le=LabelEncoder()
y=le.fit_transform(sonar['Class'])
X=sonar.drop('Class',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [24]:
l_rate = np.linspace(0.01, 0.8, 20)
n_est = [50, 100, 200]
depths = [3,5, None]
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbr = GradientBoostingClassifier(random_state=25, learning_rate=r, n_estimators=n, max_depth=d)
            gbr.fit(X_train, y_train)
            y_pred_prob = gbr.predict_proba(X_test)
            scores.append([r,n,d,log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','Log Loss'])
df_scores.sort_values(by=['Log Loss'],ascending=True)

100%|██████████| 20/20 [01:07<00:00,  3.40s/it]


,Learning Rate,N Estimator,Max Depth,Log Loss
27,0.134737,50,3.0,0.433694
18,0.093158,50,3.0,0.446665
12,0.051579,100,3.0,0.448078
45,0.217895,50,3.0,0.453574
9,0.051579,50,3.0,0.459423
...,...,...,...,...
176,0.800000,100,NaN,4.070971
173,0.800000,50,NaN,4.070971
136,0.633684,50,5.0,4.218676
142,0.633684,200,5.0,4.218676


In [29]:
l_rate = np.linspace(0.01, 0.8, 20)
n_est = [50, 100, 200]
depths = [3,5, None]
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = XGBClassifier(random_state=25, learning_rate=r, n_estimators=n, max_depth=d)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict_proba(X_test)
            scores.append([r,n,d,log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','Log Loss'])
df_scores.sort_values(by=['Log Loss'],ascending=True)

100%|██████████| 20/20 [00:24<00:00,  1.21s/it]


,Learning Rate,N Estimator,Max Depth,Log Loss
65,0.301053,50,NaN,0.422442
64,0.301053,50,5.0,0.422442
145,0.675263,50,5.0,0.429255
146,0.675263,50,NaN,0.429255
119,0.550526,50,NaN,0.431690
...,...,...,...,...
1,0.010000,50,5.0,0.577834
2,0.010000,50,NaN,0.577834
0,0.010000,50,3.0,0.588441
105,0.467368,200,3.0,0.599991


In [37]:
l_rate = np.linspace(0.01, 0.8, 20)
n_est = [50, 100, 200]
depths = [3,4,5, None]
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = LGBMClassifier(random_state=25, learning_rate=r, n_estimators=n, max_depth=d,verbose=-1)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict_proba(X_test)
            scores.append([r,n,d,log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','Log Loss'])
df_scores.sort_values(by=['Log Loss'],ascending=True)

100%|██████████| 20/20 [00:05<00:00,  3.64it/s]


,Learning Rate,N Estimator,Max Depth,Log Loss
49,0.176316,50,4.0,0.371837
37,0.134737,50,4.0,0.383137
27,0.093158,50,NaN,0.390546
26,0.093158,50,5.0,0.390546
38,0.134737,50,5.0,0.393635
...,...,...,...,...
216,0.758421,50,3.0,1.136319
199,0.675263,100,NaN,1.155843
198,0.675263,100,5.0,1.155843
203,0.675263,200,NaN,1.155843


In [41]:
l_rate = np.linspace(0.01, 0.8, 20)
n_est = [50, 100, 200]
depths = [3,4,5, None]
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = CatBoostClassifier(random_state=25, learning_rate=r, n_estimators=n, max_depth=d,verbose=0)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict_proba(X_test)
            scores.append([r,n,d,log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','Log Loss'])
df_scores.sort_values(by=['Log Loss'],ascending=True)

100%|██████████| 20/20 [06:25<00:00, 19.26s/it]


,Learning Rate,N Estimator,Max Depth,Log Loss
28,0.093158,100,3.0,0.399040
18,0.051579,100,5.0,0.401615
30,0.093158,100,5.0,0.405820
23,0.051579,200,NaN,0.408290
22,0.051579,200,5.0,0.409212
...,...,...,...,...
165,0.550526,200,4.0,0.844625
169,0.592105,50,4.0,0.859158
228,0.800000,50,3.0,0.876743
173,0.592105,100,4.0,0.880961


ValueError: Found unknown categories [np.float64(0.0067), np.float64(0.0071), np.float64(0.0089), np.float64(0.0099), np.float64(0.0107), np.float64(0.0124), np.float64(0.0129), np.float64(0.0132), np.float64(0.0134), np.float64(0.0176), np.float64(0.0188), np.float64(0.0189), np.float64(0.0192), np.float64(0.02), np.float64(0.0203), np.float64(0.0223), np.float64(0.0235), np.float64(0.0257), np.float64(0.0261), np.float64(0.0262), np.float64(0.0283), np.float64(0.0286), np.float64(0.0291), np.float64(0.0298), np.float64(0.0299), np.float64(0.0308), np.float64(0.0315), np.float64(0.0331), np.float64(0.034), np.float64(0.0346), np.float64(0.0353), np.float64(0.0374), np.float64(0.0378), np.float64(0.0409), np.float64(0.0411), np.float64(0.0423), np.float64(0.0454), np.float64(0.0459), np.float64(0.0473), np.float64(0.0519), np.float64(0.0526), np.float64(0.0629), np.float64(0.0654), np.float64(0.0707), np.float64(0.0721), np.float64(0.079), np.float64(0.0856), np.float64(0.1088), np.flo